In [43]:
import pandas as pd
import scipy
from scipy import stats
import numpy as np

from eval_functions import correctness_score, hallucination_score, hallucination_score_noramlized

vision_base_eval = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-hallucination-correctness-base.csv")

#### Scoring criteria: Correctness

- Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
- Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
- Score 2 — The generated answer is correct and captures the key facts from the reference answer.

#### Scoring criteria: Hallucination

- Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

# Vision Base Eval 20: test_1

In [44]:
vision_base_eval_20 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-20samples.csv")
vision_base_eval_20.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1d422e61-d9b9-461d-9514-d167322c325a,"{""user_query"": ""How's my form?""}","{""answer"": ""Wrists are extended back under the...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome work getting after it. Her...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,42.583386,10285,0.031829,0.5,1.0
1,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the effort here—let’s dial a ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,42.086563,19562,0.045299,1.0,1.0
2,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the backyard grind! Thanks fo...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,46.255542,10157,0.035699,0.5,1.0
3,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed your bench—le...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,69.200048,10121,0.030731,1.0,2.0
4,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed this—great st...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,45.183090,14805,0.039676,1.0,1.0


In [45]:
vision_base_eval_20.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.00000,20.000000
mean,1.0,NaN,47.563381,13743.700000,0.037699,0.65000,1.600000
std,0.0,NaN,13.011740,4295.202343,0.005575,0.28562,0.502625
min,1.0,NaN,37.114877,8450.000000,0.028813,0.00000,1.000000
25%,1.0,NaN,41.421420,10228.500000,0.033708,0.50000,1.000000
50%,1.0,NaN,44.581522,10777.000000,0.036770,0.50000,2.000000
75%,1.0,NaN,47.141303,18730.000000,0.041026,1.00000,2.000000
max,1.0,NaN,94.627459,19562.000000,0.047601,1.00000,2.000000


In [46]:
base_20_hallucination = hallucination_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")
hallucination_score_mean = vision_base_eval_20["hallucination"].mean()
print(f"Hallcination score overall {hallucination_score_mean}")


print(" ")
base_20_correctness = correctness_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")
correctness_score_mean = vision_base_eval_20["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

95% CI Hallucination Low - vision_base_eval_20_test1: 1.4
95% CI Hallucination High - vision_base_eval_20_test1: 1.8
Standard Error - Hallucination - vision_base_eval_20_test1: 0.1081
hallucination_MOE - vision_base_eval_20_test1: 0.20
Hallcination score overall 1.6
 
95% CI Correctness Low - vision_base_eval_20_test1: 1.05
95% CI Correctness High - vision_base_eval_20_test1: 1.55
Standard Error - Correctness - vision_base_eval_20_test1: 0.1267
Correctness MOE - vision_base_eval_20_test1: 0.25
Correctness score overall: 1.3


Vision-faithfulness-rag-updated-system-prompt-test3
- updated system prompt with new behavior instructions 

        # BEHAVIOR INSTRUCTIONS
        1. Tone
        - You're eager and excited to help 
        2. How to analyze
        - Break down your analysis into three sections
            What looks good
                - 1 to 2 points
            Main fixes
                - Cover all significant issues you observe
            Mental cues
                - A brief list of mental cues the lifter can easily remember during their lift

In [ ]:
vision_eval_test3 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-updated-system-prompt-test3.csv")

print("Hallucination")
vision_20_hallucination_test3 = hallucination_score_noramlized(vision_eval_test3, df_name="vision_eval_test3")
hallucination_score_mean = vision_eval_test3["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctness")
vision_20_correctness_test3 = correctness_score(vision_eval_test3, df_name="vision_eval_test3")
correctness_score_mean = vision_eval_test3["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination
95% CI Hallucination Low - vision_eval_test3: 1.35
95% CI Hallucination High - vision_eval_test3: 1.8
Standard Error - Hallucination - vision_eval_test3: 0.1101
hallucination_MOE - vision_eval_test3: 0.22
Hallcination score overall 1.6
 
Correctness
95% CI Correctness Low - vision_eval_test3: 1.3
95% CI Correctness High - vision_eval_test3: 1.7
Standard Error - Correctness - vision_eval_test3: 0.1137
Correctness MOE - vision_eval_test3: 0.20
Correctness score overall: 1.5


## Conduct t-test: baseline vs test_3: system prompt update 
- H0: Change did not have any statistically significant impact on results
- H1: Change did have a statistically significant impact on results 

In [ ]:
baseline_scores = vision_base_eval_20["hallucination"]
new_scores = vision_eval_test3["hallucination"]

res_hallucination = scipy.stats.ttest_rel(baseline_scores, new_scores)
print("Hallucination t-test:", res_hallucination)

Hallucination t-test: TtestResult(statistic=np.float64(6.838803314120882), pvalue=np.float64(1.5840256292485458e-06), df=np.int64(19))


In [58]:
baseline_scores = vision_base_eval_20["correctness"]
new_scores = vision_eval_test3["correctness"]

res_correctness = scipy.stats.ttest_rel(baseline_scores, new_scores)
print("Correctness t-test:", res_correctness)

Correctness t-test: TtestResult(statistic=np.float64(-1.2853691738343027), pvalue=np.float64(0.21411072939524578), df=np.int64(19))


## Results 

### Scores (0–2 scale)

| Metric | Baseline | Test 3 | Change |
|--------|----------|--------|--------|
| Correctness | 1.3 | 1.5 | +0.2 |
| Hallucination | 1.6 | 1.6 | 0 |

### Paired t-tests (α = 0.05)

| Metric | p-value | Significant? |
|--------|---------|-------------|
| Correctness | 0.21 | No |
| Hallucination | 1.58e-06 | Yes |

## Takeaway

Adding structured behavior instructions slightly improved correctness, though not statistically significant. No impact on hallucination.

# Model Swap Comparison: gpt-5 → gpt-5.4 (Response Generator)

In [ ]:
vision_eval_20_test_4 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-gpt-5.4-response_generator.csv")
vision_eval_20_test_4.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,24.849006,14675.300000,0.041166,0.725000,0.750000
std,0.0,NaN,4.092166,6002.343271,0.014363,0.255209,0.256495
min,1.0,NaN,20.083819,6416.000000,0.020617,0.500000,0.500000
25%,1.0,NaN,22.820954,9829.750000,0.029325,0.500000,0.500000
50%,1.0,NaN,24.168265,11795.000000,0.033617,0.500000,0.750000
75%,1.0,NaN,25.836488,21842.500000,0.057639,1.000000,1.000000
max,1.0,NaN,37.365576,22000.000000,0.059112,1.000000,1.000000


In [ ]:
print("Hallucination (test 4 - gpt-5.4)")
vision_20_hallucination_test_4 = hallucination_score_noramlized(vision_eval_20_test_4, df_name="vision_eval_test_4")
hallucination_score_mean = vision_eval_20_test_4["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctnes (test4 - gpt 5.4")
vision_20_correctness_test_4 = correctness_score(vision_eval_20_test_4, df_name="vision_eval_test_4")
correctness_score_mean = vision_eval_20_test_4["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination (test 4 - gpt-5.4)
95% CI Hallucination Low - vision_eval_test_4: 1.3
95% CI Hallucination High - vision_eval_test_4: 1.7
Standard Error - Hallucination - vision_eval_test_4: 0.1130
hallucination_MOE - vision_eval_test_4: 0.20
Hallcination score overall 1.5
 
Correctnes (test4 - gpt 5.4
95% CI Correctness Low - vision_eval_test_4: 1.25
95% CI Correctness High - vision_eval_test_4: 1.65
Standard Error - Correctness - vision_eval_test_4: 0.1115
Correctness MOE - vision_eval_test_4: 0.20
Correctness score overall: 1.45


# System prompt update: 100 word output cap

Updated:

            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and carefully. Look for issues realted to form, safety, and unhelpful camera angles.
    Be specific about what you observe and include that in your feedback.
             
        # RESPONSE STYLE
    - Lead with your most important observations in 2-3 sentences.
    - Give ONE specific, actionable cue the user can try on their next set.
    - If you notice additional areas to improve, briefly mention them so the user can ask follow-up questions.
    - Keep your total response under 100 words unless the user asks for more detail.
    - When the user asks a follow-up, answer ONLY that question. Do not repeat your full analysis.
    - If the lift looks solid overall, say so. Do not nitpick to fill space.

    # ANSWER CONTEXT
    First describe what you observe in the images.         
    Then use the ONLY following context to provide coaching advice:
 
    {top_k_chunks}
    
    If the query or image isn't in context, reply, 'I don't have expert coaching advice for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
        

In [ ]:
system_prompt_eval_v1 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-system-prompt-100-word-cap.csv")
system_prompt_eval_v1.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,9.390251,13468.300000,0.032615,0.500000,0.900000
std,0.0,NaN,1.504880,5982.938436,0.014094,0.229416,0.205196
min,1.0,NaN,7.044194,5563.000000,0.014102,0.000000,0.500000
25%,1.0,NaN,8.414615,8642.750000,0.021317,0.500000,1.000000
50%,1.0,NaN,9.372272,10466.000000,0.025466,0.500000,1.000000
75%,1.0,NaN,10.052803,20641.000000,0.049415,0.500000,1.000000
max,1.0,NaN,13.059238,20804.000000,0.050002,1.000000,1.000000


In [59]:
print("Hallucination - 100 word cap")
system_prompt_hallucination_test_1 = hallucination_score_noramlized(system_prompt_eval_v1, df_name="system_prompt_eval_v1")
hallucination_score_mean = system_prompt_eval_v1["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctnes -100 word cap")
system_prompt_correctness_test_1 = correctness_score(system_prompt_eval_v1, df_name="system_prompt_eval_v1")
correctness_score_mean = system_prompt_eval_v1["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination - 100 word cap
95% CI Hallucination Low - system_prompt_eval_v1: 1.55
95% CI Hallucination High - system_prompt_eval_v1: 1.95
Standard Error - Hallucination - system_prompt_eval_v1: 0.0891
hallucination_MOE - system_prompt_eval_v1: 0.20
Hallcination score overall 1.8
 
Correctnes -100 word cap
95% CI Correctness Low - system_prompt_eval_v1: 0.8
95% CI Correctness High - system_prompt_eval_v1: 1.2
Standard Error - Correctness - system_prompt_eval_v1: 0.1008
Correctness MOE - system_prompt_eval_v1: 0.20
Correctness score overall: 1.0


## Compare system prompt update to previous baseline using t-test

- H0: prompt change did not have any statistically significant impact on results
- H1: prompt change did have a statistically significant impact on results 

### Correctness

In [ ]:
baseline_scores = vision_base_eval_20["correctness"]
system_prompt_v1_scores = system_prompt_eval_v1["correctness"]

correctness_result = scipy.stats.ttest_rel(baseline_scores, system_prompt_v1_scores)
correctness_result

TtestResult(statistic=np.float64(1.674299462021233), pvalue=np.float64(0.11045413930370163), df=np.int64(19))

#### Hallucination

In [ ]:
baseline_scores = vision_base_eval_20["hallucination"]
system_prompt_v1_scores = system_prompt_eval_v1["hallucination"]

hallucination_result = scipy.stats.ttest_rel(baseline_scores, system_prompt_v1_scores)
hallucination_result

TtestResult(statistic=np.float64(5.9839528998557725), pvalue=np.float64(9.290522475392668e-06), df=np.int64(19))

#### Results
- Correctness: p-value > 0.05. Cannot reject the null hypothesis. Prompt change did NOT have a statistically significant impact on results.
- Hallucination: p-value < 0.05. Reject the null hypothesis. Prompt change DID have a statistically significant impact on results.


#### Outcome: 

The concise prompt significantly reduced hallucination (p < 0.001) while the correctness change was not statistically significant (p = 0.11). Meanwhile,  we got a better, faster response without a proven cost to accuracy.

# Fitness Form Coach — Optimization Results

## 1. Model Swap: gpt-4o-mini → gpt-5.4-nano

### Nodes Changed
- Router (query classification)
- Chat memory (conversational responses)
- Classifier (exercise classification)

### Latency Comparison

| Query | Before (gpt-4o-mini) | After (gpt-5.4-nano) | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | 8.21s | 5.32s | -35% |
| "can you tell me again..." (follow-up) | 15.95s | 8.08s | -49% |
| Bench press text query (vector_db → response_gen) | 43.63s | 42.02s | -4% |
| Video query (full pipeline) | 103.80s | 58.11s | -44% |

### Cost Comparison

| Query | Before cost | After cost | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | $0.0452 | $0.0012 | -97% |
| "can you tell me again..." (follow-up) | $0.0474 | $0.0040 | -92% |
| Bench press text query | $0.0350 | $0.0301 | -14% |
| Video query (full pipeline) | $0.0500 | $0.0452 | -10% |

### Key Takeaways
- Chat memory and follow-up queries saw the biggest improvements: ~35-49% latency reduction and ~90-97% cost reduction.
- Video query latency nearly halved (103.8s → 58.1s), driven by the classifier switch to nano.
- Text queries routed through vector_db → response_generator barely changed (~4%) because the bottleneck is the response_generator LLM — not yet optimized at this stage.

---

## 2. Memory Summarization

### Change
- Removed `RunnableWithMessageHistory` from `response_generator`
- Manually manage history: save summarized response (via gpt-5.4-nano) instead of full video analysis
- User still sees the full response; history stores a ~150-word summary
- Summarization prompt extracts: exercise, main issues, priority fixes, uncertainties, and a concise coaching summary

### Results (same session: video → text query → follow-up → thanks)

| Query | Before (full history) | After (summarized) | Change |
|---|---|---|---|
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 4.52s | 1.74s | -62% |
| Text follow-up tokens | 18,100 | 2,863 | -84% |
| Text follow-up latency | — | 4.48s | — |
| "can you tell me again..." tokens | — | 1,372 | — |

### Key Takeaways
- Token usage on follow-up messages dropped 84-98% because history no longer contains the full video analysis.
- Response quality on follow-ups is unchanged — the summary retains enough context for the model to answer questions about tempo, grip, form cues, etc.
- The summarization adds a one-time nano LLM call after each video analysis, but every subsequent message in the session benefits from the reduced history size.

---

## 3. Model Swap: GPT-5 → GPT-5.4 (Response Generator)

### Node Changed
- Response generator (video analysis)

### Quality Comparison (LangSmith evals, n=20)

| Metric | GPT-5 (baseline) | GPT-5.4 | Change |
|---|---|---|---|
| Correctness | 0.75 | 0.72 | -0.03 (not significant) |
| Hallucination | 0.80 | 0.75 | +0.05 (not significant) |

### Latency Comparison

| Metric | GPT-5 | GPT-5.4 | Change |
|---|---|---|---|
| P50 latency | 53.71s | 24.17s | -55% |
| P99 latency | 103.13s | 35.93s | -65% |

### Key Takeaways
- Correctness and hallucination scores are effectively unchanged — the -0.03 and +0.05 differences are well within the margin of error for 20 samples.
- P50 latency dropped 55%, P99 dropped 65%. The video analysis pipeline is now more than twice as fast.
- GPT-5.4 is a clear win: same quality, dramatically faster.

---

## Cumulative Impact

| Metric | Original | Final | Total Change |
|---|---|---|---|
| Video analysis latency (P50) | 103.80s | 24.17s | -77% |
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 8.21s | 1.74s | -79% |
| "thanks for your help" cost | $0.0452 | $0.0001 | -99% |
| Follow-up query tokens | 18,100 | 1,372 | -92% |